In [4]:
import sys, subprocess

pkgs = [
    [sys.executable, "-m", "pip", "install", "-q",
     "--force-reinstall", "--no-cache-dir", "pyarrow"],
    [sys.executable, "-m", "pip", "install", "-q",
     "setfit==1.1.0", "datasets<3.0.0", "accelerate", "sentence-transformers"],
]
for cmd in pkgs:
    r = subprocess.run(cmd, capture_output=True, text=True)
    label = " ".join(cmd[4:6])
    if r.returncode != 0:
        print(f"❌ {label}: {r.stderr[-200:]}")
    else:
        print(f"✅ {label}")

# Cek versi transformers yg terinstall
r = subprocess.run([sys.executable, "-c",
    "import transformers; print('transformers:', transformers.__version__)"],
    capture_output=True, text=True)
print(r.stdout.strip())
print("✅ Selesai - restart kernel dilanjutkan otomatis")

✅ -q --force-reinstall
✅ -q setfit==1.1.0
transformers: 5.3.0
✅ Selesai - restart kernel dilanjutkan otomatis


In [1]:
import os, gc, warnings, socket, requests
import numpy as np
import pandas as pd
import torch
from datetime import datetime

warnings.filterwarnings("ignore")
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# === PATCH 1: default_logdir (dihapus di transformers ≥ 4.45) ===
import transformers.training_args as _ta
if not hasattr(_ta, "default_logdir"):
    def default_logdir() -> str:
        t = datetime.now().strftime("%b%d_%H-%M-%S")
        return os.path.join("runs", t + "_" + socket.gethostname())
    _ta.default_logdir = default_logdir
    print("🔧 Patch 1: default_logdir")

# === PATCH 2: Block ModelCard callback ===
import transformers.trainer_callback as _tcb
if not getattr(_tcb.CallbackHandler.add_callback, "_is_patched", False):
    _orig_add_cb = _tcb.CallbackHandler.add_callback
    def _safe_add_cb(self, cb):
        if "ModelCard" in str(type(cb)):
            return
        return _orig_add_cb(self, cb)
    _safe_add_cb._is_patched = True
    _tcb.CallbackHandler.add_callback = _safe_add_cb
    print("🔧 Patch 2: ModelCard callback diblokir")

# === PATCH 4: Tambahkan attr yg hilang di CallbackHandler (transformers 5.x) ===
if not getattr(_tcb.CallbackHandler, "_setfit_compat", False):
    _orig_cb_init = _tcb.CallbackHandler.__init__
    def _compat_cb_init(self, *args, **kwargs):
        _orig_cb_init(self, *args, **kwargs)
        for attr in ("tokenizer", "optimizer", "lr_scheduler", "train_dataloader", "eval_dataloader"):
            if not hasattr(self, attr):
                setattr(self, attr, None)
    _tcb.CallbackHandler.__init__ = _compat_cb_init
    _tcb.CallbackHandler._setfit_compat = True
    print("🔧 Patch 4: CallbackHandler compat attrs")

# === Import setfit ===
try:
    import setfit.trainer as _st
    _st.BCSentenceTransformersTrainer.on_init_end = lambda *a, **kw: None
except Exception:
    pass

from setfit import SetFitModel, Trainer, TrainingArguments

# === PATCH 3: Fix hf_hub_download - idempotent ===
import setfit.modeling as _sm
if not getattr(_sm.hf_hub_download, "_is_patched", False):
    _real_hf_dl = _sm.hf_hub_download
    def _patched_hf_dl(*args, **kwargs):
        try:
            return _real_hf_dl(*args, **kwargs)
        except Exception as e:
            msg = str(e)
            if any(x in msg for x in ["404", "Not Found", "EntryNotFound", "RemoteEntryNotFound"]):
                raise requests.exceptions.HTTPError(f"404: {msg}")
            raise
    _patched_hf_dl._is_patched = True
    _sm.hf_hub_download = _patched_hf_dl
    print("🔧 Patch 3: hf_hub_download")

# === PATCH 5: Fix BCSentenceTransformersTrainer.log() signature (transformers 5.x tambah start_time arg) ===
import setfit.trainer as _st2
if not getattr(_st2.BCSentenceTransformersTrainer.log, "_is_patched", False):
    _orig_bcs_log = _st2.BCSentenceTransformersTrainer.log
    def _patched_bcs_log(self, logs, *args, **kwargs):
        return _orig_bcs_log(self, logs)
    _patched_bcs_log._is_patched = True
    _st2.BCSentenceTransformersTrainer.log = _patched_bcs_log
    print("🔧 Patch 5: BCSentenceTransformersTrainer.log() signature fixed")

# === CleanTrainer ===
class CleanTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        kwargs["callbacks"] = []
        super().__init__(*args, **kwargs)
        if hasattr(self, "callback_handler"):
            self.callback_handler.callbacks = []

    def _validate_column_mapping(self, dataset):
        pass  # Bypass - plain dict tidak punya .column_names

# === CEK GPU ===
if torch.cuda.is_available():
    device = "cuda"
    torch.cuda.empty_cache()
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = "cpu"
    print("⚠️  CPU mode")

print(f"   Device: {device}")
print("✅ Semua patch & import selesai.")

🔧 Patch 1: default_logdir
🔧 Patch 2: ModelCard callback diblokir
🔧 Patch 4: CallbackHandler compat attrs
🔧 Patch 3: hf_hub_download
🔧 Patch 5: BCSentenceTransformersTrainer.log() signature fixed
✅ GPU: NVIDIA GeForce RTX 4070 Ti
   VRAM: 12.9 GB
   Device: cuda
✅ Semua patch & import selesai.


In [2]:
# --- LOAD DATA ---
BASE      = os.path.join(os.path.dirname(os.path.abspath("__file__")),
                         r"c:\Users\FATISDA UNS\Documents\Reni\preprocessing")
DATA_PATH = r"c:\Users\FATISDA UNS\Documents\Reni\preprocessing\clean_dataset\annotator3_clean.csv"
OUT_PATH  = r"c:\Users\FATISDA UNS\Documents\Reni\preprocessing\clean_dataset\annotator3_clean_final.csv"

df = pd.read_csv(DATA_PATH, dtype=str).fillna("")

# Split: sudah terlabeli = training, belum = prediksi
mask_labeled = (df["category"] != "") & (df["sentiment"] != "")
df_train = df[mask_labeled].copy().reset_index(drop=True)
df_pred   = df[~mask_labeled].copy().reset_index(drop=True)

print(f"Total baris    : {len(df)}")
print(f"Training data  : {len(df_train)} baris (labeled)")
print(f"Prediksi data  : {len(df_pred)} baris (unlabeled)")
print()
print("Distribusi category (train):")
print(df_train["category"].value_counts().to_string())
print()
print("Distribusi sentiment (train):")
print(df_train["sentiment"].value_counts().to_string())
print()
print("Baris terakhir training:")
print(df_train.tail(3)[["sentence_id","category","sentiment"]].to_string(index=False))
print("Baris pertama prediksi:")
print(df_pred.head(3)[["sentence_id","category","sentiment"]].to_string(index=False))

# --- MAPPING LABEL ---
cat_list  = ["none", "app", "service", "app dan service"]
sent_list = ["negative", "neutral", "positive"]

cat2id  = {v: i for i, v in enumerate(cat_list)}
id2cat  = {i: v for i, v in enumerate(cat_list)}
sent2id = {v: i for i, v in enumerate(sent_list)}
id2sent = {i: v for i, v in enumerate(sent_list)}

df_train["label_cat"]  = df_train["category"].map(cat2id).astype(int)
df_train["label_sent"] = df_train["sentiment"].map(sent2id).astype(int)

train_texts = df_train["content"].astype(str).tolist()

# Gunakan plain dict (bukan datasets.Dataset) agar tidak butuh pyarrow
# SetFit Trainer mendukung format ini karena internally: dataset["text"], dataset["label"]
ds_cat  = {"text": train_texts, "label": df_train["label_cat"].tolist()}
ds_sent = {"text": train_texts, "label": df_train["label_sent"].tolist()}

print("\n✅ Data siap (plain dict, tanpa pyarrow dependency).")

Total baris    : 38345
Training data  : 19998 baris (labeled)
Prediksi data  : 18347 baris (unlabeled)

Distribusi category (train):
category
none               8868
app                5735
service            4839
app dan service     556

Distribusi sentiment (train):
sentiment
negative    8565
positive    8272
neutral     3161

Baris terakhir training:
   sentence_id category sentiment
review_13322_1     none   neutral
review_13322_2     none  positive
review_13323_1     none  positive
Baris pertama prediksi:
   sentence_id category sentiment
review_13324_1                   
review_13325_1                   
review_13326_1                   

✅ Data siap (plain dict, tanpa pyarrow dependency).


In [3]:
# --- TRAINING MODEL KATEGORI ---
# Model multilingual terbaik untuk teks Indonesia
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

print("=" * 55)
print("  TRAINING MODEL KATEGORI")
print("=" * 55)
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

model_cat = SetFitModel.from_pretrained(MODEL_NAME)
model_cat.to(device)

args_cat = TrainingArguments(
    batch_size           = 32,      # optimal untuk RTX 4070 Ti
    num_epochs           = 3,       # epoch fine-tuning sentence transformer
    num_iterations       = 20,      # pair per class (cukup karena data 20k)
    body_learning_rate   = 2e-5,    # rate backbone
    head_learning_rate   = 1e-2,    # rate classifier
    end_to_end           = True,    # fine-tune seluruh model
    use_amp              = (device == "cuda"),   # mixed precision (FP16) di GPU
    report_to            = "none",
)

trainer_cat = CleanTrainer(
    model          = model_cat,
    train_dataset  = ds_cat,   # plain dict {"text": [...], "label": [...]}
    args           = args_cat,
    # column_mapping tidak dipakai karena key sudah "text" & "label"
)
trainer_cat.train()

print("\n✅ Training kategori selesai!")
model_cat.save_pretrained("setfit_model_category")
print("   Model disimpan → setfit_model_category/")

  TRAINING MODEL KATEGORI


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.
***** Running training *****
  Num unique pairs = 799920
  Batch size = 32
  Num epochs = 3


Step,Training Loss
1,0.258556
50,0.186949
100,0.166125
150,0.152793
200,0.147314
250,0.137530
300,0.131689
350,0.122487
400,0.122357
450,0.120646


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Training kategori selesai!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

   Model disimpan → setfit_model_category/


In [4]:
# --- TRAINING MODEL SENTIMEN ---
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

print("=" * 55)
print("  TRAINING MODEL SENTIMEN")
print("=" * 55)

# Aktifkan expandable memory segments (kurangi fragmentasi VRAM)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Bebaskan VRAM dari model kategori jika ada di memori
try:
    del model_cat
    del trainer_cat
except NameError:
    pass
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f"   VRAM bebas: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

model_sent = SetFitModel.from_pretrained(MODEL_NAME)
model_sent.to(device)

args_sent = TrainingArguments(
    batch_size           = 16,    # lebih aman untuk 12GB VRAM
    num_epochs           = 3,
    num_iterations       = 20,
    body_learning_rate   = 2e-5,
    head_learning_rate   = 1e-2,
    end_to_end           = True,
    use_amp              = (device == "cuda"),
    report_to            = "none",
)

trainer_sent = CleanTrainer(
    model          = model_sent,
    train_dataset  = ds_sent,
    args           = args_sent,
)
trainer_sent.train()

print("\n✅ Training sentimen selesai!")
model_sent.save_pretrained("setfit_model_sentiment")
print("   Model disimpan → setfit_model_sentiment/")

  TRAINING MODEL SENTIMEN
   VRAM bebas: 11.6 GB


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.
***** Running training *****
  Num unique pairs = 799920
  Batch size = 16
  Num epochs = 3


Step,Training Loss
1,0.244811
50,0.166541
100,0.142618
150,0.115751
200,0.117364
250,0.120251
300,0.119384
350,0.116952
400,0.106666
450,0.112032


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Training sentimen selesai!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

   Model disimpan → setfit_model_sentiment/


In [ ]:
# --- PREDIKSI SISA DATA & SIMPAN ---
print("=" * 55)
print(f"  PREDIKSI {len(df_pred)} BARIS")
print("=" * 55)

gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

CAT_MODEL_PATH  = os.path.abspath("setfit_model_category")
SENT_MODEL_PATH = os.path.abspath("setfit_model_sentiment")

# === Patch infer_st_id yang crash jika _name_or_path=None (saat load dari lokal) ===
import setfit.model_card as _mc
if not getattr(_mc.SetFitModelCardData.infer_st_id, "_is_patched", False):
    _orig_infer = _mc.SetFitModelCardData.infer_st_id
    def _safe_infer_st_id(self, setfit_model_id):
        try:
            _orig_infer(self, setfit_model_id)
        except Exception:
            pass
    _safe_infer_st_id._is_patched = True
    _mc.SetFitModelCardData.infer_st_id = _safe_infer_st_id
    print("🔧 Patch 6: infer_st_id error bypass aktif")

# Load kedua model dari lokal
print(f"   Load model kategori dari {CAT_MODEL_PATH}...")
model_cat = SetFitModel.from_pretrained(CAT_MODEL_PATH, local_files_only=True)
model_cat.to(device)

print(f"   Load model sentimen dari {SENT_MODEL_PATH}...")
model_sent = SetFitModel.from_pretrained(SENT_MODEL_PATH, local_files_only=True)
model_sent.to(device)

print(f"   Mulai prediksi {len(df_pred)} baris...")

pred_texts    = df_pred["content"].astype(str).tolist()
BATCH         = 512
pred_cat_ids  = []
pred_sent_ids = []

for i in range(0, len(pred_texts), BATCH):
    batch = pred_texts[i : i + BATCH]
    pred_cat_ids.extend(model_cat.predict(batch).tolist())
    pred_sent_ids.extend(model_sent.predict(batch).tolist())
    done = min(i + BATCH, len(pred_texts))
    if (i // BATCH + 1) % 5 == 0 or done == len(pred_texts):
        print(f"   {done}/{len(pred_texts)} ({done/len(pred_texts)*100:.1f}%)")

df_pred_out = df_pred.copy()
df_pred_out["category"]  = [id2cat[int(p)]  for p in pred_cat_ids]
df_pred_out["sentiment"] = [id2sent[int(p)] for p in pred_sent_ids]

# Gabungkan dan kembalikan ke urutan asli
labeled_idx   = df[mask_labeled].index.tolist()
unlabeled_idx = df[~mask_labeled].index.tolist()

df_result = pd.concat(
    [df_train.drop(columns=["label_cat", "label_sent"]), df_pred_out],
    ignore_index=True
)
df_result["_orig_idx"] = labeled_idx + unlabeled_idx
df_result = df_result.sort_values("_orig_idx").drop(columns=["_orig_idx"]).reset_index(drop=True)

# Simpan
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
df_result.to_csv(OUT_PATH, index=False)

print(f"\n✅ SELESAI! Disimpan ke: {OUT_PATH}")
print(f"   Total baris      : {len(df_result)}")
print(f"   Empty category   : {(df_result['category']=='').sum()}")
print(f"   Empty sentiment  : {(df_result['sentiment']=='').sum()}")
print()
print("Distribusi category (final):")
print(df_result["category"].value_counts().to_string())
print()
print("Distribusi sentiment (final):")
print(df_result["sentiment"].value_counts().to_string())
df_result.tail(5)

  PREDIKSI 18347 BARIS
🔧 Patch 6: infer_st_id error bypass aktif
   Load model kategori dari c:\Users\FATISDA UNS\Documents\Reni\preprocessing\setfit_model_category...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   Load model sentimen dari c:\Users\FATISDA UNS\Documents\Reni\preprocessing\setfit_model_sentiment...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   Mulai prediksi 18347 baris...
   2560/18347 (14.0%)
   5120/18347 (27.9%)
   7680/18347 (41.9%)
   10240/18347 (55.8%)
   12800/18347 (69.8%)
   15360/18347 (83.7%)
   17920/18347 (97.7%)
   18347/18347 (100.0%)

✅ SELESAI! Disimpan ke: c:\Users\FATISDA UNS\Documents\Reni\preprocessing\clean_dataset\annotator3_clean_final.csv
   Total baris      : 38345
   Empty category   : 0
   Empty sentiment  : 0

Distribusi category (final):
category
none               17522
app                10554
service             9261
app dan service     1008

Distribusi sentiment (final):
sentiment
positive    16636
negative    15949
neutral      5760


,sentence_id,at,content,category,sentiment
38340,review_25740_1,2025-05-01 01:11:41,trimakasih bnyk sngat membantu sekali jadi nga...,service,positive
38341,review_25741_1,2025-05-01 01:07:55,ok,none,neutral
38342,review_25742_1,2025-05-01 00:21:23,mudah digunakan,none,positive
38343,review_25743_1,2025-05-01 00:21:07,Baru kali ini isi saldo gopay ga masuk pdhal l...,app,negative
38344,review_25743_2,2025-05-01 00:21:07,Malah dsuruh tunggu 2x24 jam 😭,service,negative


: 